## (Optional) Install Setup Packages
In order to install the packages used to import the test data, please run the
following command in your terminal from the root directory of the repository:

```
pip install -r scripts/python/requirements.txt
```

## Load Test Dataset Into Memory

In [16]:
import polars as pl
from pathlib import Path

# Load the test data
test_file = Path("../../test-data.xlsx")

# Read Excel file with multiple sheets
loans_raw = (pl
    .read_excel(test_file, sheet_name="loans")
    .rename(lambda col: col.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""))
)
cashflows_raw = (pl
    .read_excel(test_file, sheet_name="cash flows")
    .rename(lambda col: col.lower().replace(" ", "_").replace("-", "_").replace("(", "").replace(")", ""))
)


## Inspect Data


In [19]:
loans_raw.describe()

statistic,customer_id,year_of_birth,gender,contract_number,product_category,loan_type,number_of_customer_active_contracts,customer_active_contract_numbers,termination_date,termination_reason,interest_rate,loan_amount,sum_of_last_payment,debt_principal,debt_interest,total_outstanding_and_debt_amount,proceeding_start_date,delay_days_principal
str,f64,f64,str,str,str,str,str,str,str,str,f64,str,f64,f64,f64,f64,str,f64
"""count""",552.0,552.0,"""552""","""552""","""552""","""552""","""552""","""552""","""552""","""552""",551.0,"""552""",552.0,552.0,552.0,552.0,"""552""",552.0
"""null_count""",0.0,0.0,"""0""","""0""","""0""","""0""","""0""","""0""","""0""","""0""",1.0,"""0""",0.0,0.0,0.0,0.0,"""0""",0.0
"""mean""",1.4798e6,78.282609,null,null,null,null,null,null,null,null,11.2102,null,1261.405942,66894.469112,2279.625507,69174.09462,null,621.708333
"""std""",923684.210521,11.973346,null,null,null,null,null,null,null,null,3.236196,null,1992.811062,58343.966576,2124.730365,60018.709053,null,161.879826
"""min""",351510.0,0.0,"""Female""","""1647150/90ip-1""","""Consumer loan""","""LOANSE01""","""1""","""1708513/90ip""","""2019-03-08""","""Debt Relief""",0.075,"""10000""",-2300.0,122.8,0.0,122.8,"""2023-03-09""",341.0
"""25%""",551631.0,70.0,null,null,null,null,null,null,null,null,8.5,null,264.12,24466.15,591.69,25222.93,null,483.0
"""50%""",1.191975e6,80.0,null,null,null,null,null,null,null,null,10.2,null,919.0,42710.28,1827.57,44790.87,null,623.0
"""75%""",2.407682e6,87.0,null,null,null,null,null,null,null,null,14.75,null,1725.14,103485.6,3271.67,106891.03,null,762.0
"""max""",2.803295e6,99.0,"""Male""","""SL-SEC-19002806""","""Consumer loan""","""SMALL_LOAN_REVOLVING_SE01""","""4""","""SL-SEC-19002806, CL-SEC-200014…","""2025-10-13 00:00:00""","""Debts delay 90+ days""",23.95,"""99386.29""",33586.0,448285.91,14769.0,463054.91,"""2025-09-10""",901.0


In [ ]:
cashflows_raw.describe()

## Task 1 - Restructure Loan Data

The client has provided us with the loan data in a format that is not compatible with our internal loan data schema. In particular, the data provided by the client is a single table that contains the characteristics for both the loans and the borrowers.

Our internal schema specifies that the loan and borrower details should belong in separate tables with an additional relationship table linking them both based on their respective primary ids.

The desired schema for the loan information is described below, where each key-value pair represents the field name-type belonging to each table:


In [13]:
# Define the target schema
target_loan_schema = {
    "loans": {
        "contract_number": "str",  # PRIMARY KEY
        "termination_date": "date",
        "termination_reason": "str",
        "interest_rate": "float64",
        "loan_amount": "float64",
        "sum_of_last_payment": "float64",
        "debt_principal": "float64",
        "debt_interest": "float64",
        "total_outstanding_and_debt_amount": "float64",
        "proceeding_start_date": "date",
        "delay_days_principal": "int64"
    },
    "borrowers": {
        "customer_id": "str",  # PRIMARY KEY
        "year_of_birth": "int64",
        "gender": "str",
        "number_of_customer_active_contracts": "int64",
        "customer_active_contract_numbers": "str"
    },
    "id_loan_borrower": {
        "contract_number": "str",  # PRIMARY KEY
        "customer_id": "str"
    }
}


## Task 1 - Solution

In [ ]:
def convert_to_date(series):
    """Convert string dates to proper date format"""
    return pl.Series(series).str.to_date("%Y-%m-%d", strict=False)

def convert_double(series):
    """Convert string numbers with comma decimal separator to float"""
    if series.dtype in [pl.Float64, pl.Float32, pl.Int64, pl.Int32]:
        return series
    return series.str.replace(",", ".").cast(pl.Float64)

loans = (loans_raw
    .select(target_loan_schema["loans"].keys())
    .unique(subset=["contract_number"])  # Remove duplicate contract numbers
    .with_columns([
        # Convert string fields
        pl.col("contract_number").cast(pl.Utf8),
        pl.col("termination_reason").cast(pl.Utf8),

        # Convert integer fields
        pl.col("delay_days_principal").cast(pl.Int64),

        # Convert date fields
        pl.col("termination_date").str.to_date("%Y-%m-%d", strict=False),
        pl.col("proceeding_start_date").str.to_date("%Y-%m-%d", strict=False),
    ])
)


In [ ]:
# Create borrowers table
borrowers = (loans_raw
    .select(target_loan_schema["borrowers"].keys())
    .unique(subset=["customer_id"])  # Remove duplicate customer IDs
    .with_columns([
        # Convert string fields
        pl.col("customer_id").cast(pl.Utf8),
        pl.col("gender").cast(pl.Utf8),
        pl.col("customer_active_contract_numbers").cast(pl.Utf8),

        # Convert integer fields
        pl.col("year_of_birth").cast(pl.Int64),
        pl.col("number_of_customer_active_contracts").cast(pl.Int64)
    ])
)


In [22]:
# Create loan-borrower relationship table
id_loan_borrower = (loans_raw
    .select(target_loan_schema["id_loan_borrower"].keys())
    .unique(subset=["contract_number"])  # Remove duplicate contract numbers
    .with_columns([
        pl.col("contract_number").cast(pl.Utf8),
        pl.col("customer_id").cast(pl.Utf8)
    ])
)

## Task 2 - Restructure Cashflow Data

Similar to the loan data, the client has provided the cash flow information in a format that is incompatible with our internal cash flow schema. Specifically, the client has provided the data in a "wide" format where each cash flow date is a column. We require the data to be in a "long" format according to the following schema:


In [23]:
# Define the target cashflow schema
target_cashflow_schema = {
    "cashflows": {
        "contract_number": "str",  # PRIMARY KEY - Component 1
        "cashflow_date": "date",   # PRIMARY KEY - Component 2
        "cashflow_amount": "float64"
    }
}

## Task 2 - Solution

In [ ]:
from datetime import datetime, timedelta

def format_dates(date_strings):
    """Convert date strings like 'x2023_01' to proper dates (last day of month)"""
    formatted_dates = []
    for date_str in date_strings:
        clean_date = date_str + '_01'
        parsed_date = datetime.strptime(clean_date, '%Y_%m_%d')
        if parsed_date.month == 12:
            last_day = parsed_date.replace(year=parsed_date.year + 1, month=1, day=1) - timedelta(days=1)
        else:
            last_day = parsed_date.replace(month=parsed_date.month + 1, day=1) - timedelta(days=1)
        formatted_dates.append(last_day.date())
    return formatted_dates

date_columns = [col for col in cashflows_raw.columns if col.startswith('x20')]

cashflows = (cashflows_raw
    .rename({"contract_nr": "contract_number"})  # Make loan id field name consistent
    .unpivot(
        index=["contract_number"],
        on=date_columns,
        variable_name="cashflow_date",
        value_name="cashflow_amount"
    )
    .with_columns([
        pl.col("cashflow_date").map_elements(
            lambda x: format_dates([x])[0] if x else None,
            return_dtype=pl.Date
        ),
        pl.col("cashflow_amount").cast(pl.Float64)
    ])
    .filter(pl.col("cashflow_amount").is_not_null())  # Remove rows with null amounts
)

cashflows

## Task 3 - Calculated Fields

Although the client data set contains all of the data that we require, it is lacking certain fields that are needed by subsequent data processing pipelines. In particular, we need to generate the following fields based on the existing fields we have available:

1. **Cumulative Cash Flow Amounts** - For each loan we need to have a field that contains the sum of all cash flow amounts received to date.

2. **Days Past Due (Buckets)** - For each loan we require a field that classifies the loan based on whether the field `delay_days_principal` falls into one of the following buckets:
         [0, 180), [180, 360), [360, 540), [540, 720), 720+

## Task 3 - Solution

In [ ]:
# Calculate cumulative cashflow amounts for each loan
cumulative_cashflow = (cashflows
    .group_by("contract_number")
    .agg(pl.col("cashflow_amount").sum().alias("cumulative_cashflow_amount"))
)


In [ ]:
# Create days past due buckets and join with cumulative cashflow
loans = (loans
    .join(cumulative_cashflow, on="contract_number", how="left")
    .with_columns([
        pl.col("delay_days_principal").cut(
        breaks=[180, 360, 540, 720],
        labels=["[0, 180)", "[180, 360)", "[360, 540)", "[540, 720)", "720+"],
        left_closed=True
    ).alias("days_past_due_bucket")
    ])
)